# 🗄️ Session 4: Vector Databases & RAG Pipelines
### Agentic AI Nano Bootcamp | Day 1 · Session 4

---

## 🎯 Learning Objectives
By the end of this session you will be able to:
- Explain what vector embeddings are and why they matter
- Set up and query ChromaDB as a local vector store
- Build a complete RAG (Retrieval-Augmented Generation) pipeline
- Ingest a CSV dataset into a vector database
- Answer questions using retrieved context + LLM generation
- Build a document retrieval system with LangChain

## 📋 Session Outline
1. What are Vector Embeddings?
2. How Vector Databases Work
3. Similarity Search
4. What is RAG?
5. 🔬 Lab 1: CSV → ChromaDB (Olympics Dataset)
6. 🔬 Lab 2: Query & Retrieve Answers from Vector DB
7. 🔬 Lab 3: Full RAG Pipeline with LangChain

---

In [ ]:
# ── Setup: Install dependencies ──
import subprocess
print('📦 Installing dependencies (may take 1-2 minutes)...')
pkgs = ['openai', 'chromadb', 'langchain', 'langchain-openai',
        'langchain-community', 'python-dotenv', 'pandas', 'matplotlib', 'scikit-learn']
subprocess.run(['pip', 'install'] + pkgs + ['-q'], capture_output=True)
print('✅ All dependencies installed')

import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI
client = OpenAI()
print('✅ OpenAI client ready')

## 1. What are Vector Embeddings?

An **embedding** is a dense numerical vector (list of floating-point numbers) that represents the *semantic meaning* of a piece of text.

```
Text: "The internet connection is down"
      ↓  Embedding Model  ↓
Vector: [0.023, -0.418, 0.891, 0.034, -0.209, ... ]  (1536 numbers)
```

### The Key Property: Semantic Similarity = Vector Closeness

```
"internet is broken"       →  vector A
"broadband not working"    →  vector B   ← very close to A (similar meaning)
"my cat is sleeping"       →  vector C   ← very far from A (different meaning)
```

Traditional keyword search misses synonyms. Vector search finds **semantically similar** content even with completely different words.

### Embedding Models

| Model | Dimensions | Best For |
|---|---|---|
| `text-embedding-3-small` | 1536 | General purpose, fast, cheap |
| `text-embedding-3-large` | 3072 | Higher accuracy |
| `sentence-transformers/all-MiniLM-L6-v2` | 384 | Free, open-source |
| `BAAI/bge-large-en` | 1024 | SOTA open-source |

In [ ]:
# ── Embedding Demo: Visualizing Semantic Similarity ──

import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

def get_embedding(text, model='text-embedding-3-small'):
    """Get embedding vector for a text string."""
    response = client.embeddings.create(input=text, model=model)
    return np.array(response.data[0].embedding)

# Sample texts from telecom domain
texts = [
    # Connectivity issues
    "My internet connection is not working",
    "Broadband service is down since morning",
    "Cannot connect to the internet",
    "Network outage in my area",
    # Billing issues
    "My bill is higher than expected this month",
    "I was overcharged on my last invoice",
    "Unexpected charges on my account",
    # Unrelated
    "The weather is nice today",
    "I enjoy cooking pasta for dinner",
]

print('🔢 Generating embeddings...')
embeddings = np.array([get_embedding(t) for t in texts])
print(f'✅ Generated {len(embeddings)} embeddings of dimension {embeddings.shape[1]}')

# Reduce to 2D for visualization
pca = PCA(n_components=2)
embeddings_2d = pca.fit_transform(embeddings)

# Plot
colors = ['#1F4E79']*4 + ['#1B5E3B']*3 + ['#C0392B']*2
labels = ['Connectivity']*4 + ['Billing']*3 + ['Unrelated']*2

fig, ax = plt.subplots(figsize=(12, 7))
for i, (x, y) in enumerate(embeddings_2d):
    ax.scatter(x, y, c=colors[i], s=120, zorder=3)
    ax.annotate(texts[i][:35], (x, y), textcoords='offset points',
                xytext=(8, 4), fontsize=8.5, color=colors[i])

from matplotlib.patches import Patch
legend = [Patch(color='#1F4E79', label='Connectivity issues'),
          Patch(color='#1B5E3B', label='Billing issues'),
          Patch(color='#C0392B', label='Unrelated')]
ax.legend(handles=legend, fontsize=10)
ax.set_title('Embedding Space: Semantic Clusters (PCA to 2D)', fontsize=13, fontweight='bold')
ax.set_xlabel('PCA Component 1'); ax.set_ylabel('PCA Component 2')
ax.spines[['top','right']].set_visible(False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()
print('\n💡 Notice: similar topics cluster together even with different words!')

In [ ]:
# ── Cosine Similarity: Measuring Text Closeness ──

from sklearn.metrics.pairwise import cosine_similarity

query = "internet not working"
candidates = [
    "broadband outage detected",
    "cannot access websites",
    "billing query about last month",
    "router light is blinking red",
    "upgrade to 5G plan",
    "internet is broken since yesterday",
]

query_vec = get_embedding(query).reshape(1, -1)
cand_vecs = np.array([get_embedding(c) for c in candidates])
similarities = cosine_similarity(query_vec, cand_vecs)[0]

# Sort by similarity
ranked = sorted(zip(similarities, candidates), reverse=True)

print(f'🔍 Query: "{query}"')
print('\nRanked by cosine similarity:')
print(f'{"Rank":<6} {"Score":<8} {"Text"}')
print('-' * 60)
for rank, (score, text) in enumerate(ranked, 1):
    bar = '█' * int(score * 30)
    print(f'{rank:<6} {score:.4f}   {text}')
    print(f'       {bar}')

print('\n💡 Cosine similarity: 1.0 = identical meaning, 0.0 = completely different')

## 2. How Vector Databases Work

A vector database stores embeddings alongside their original content and metadata, enabling fast **approximate nearest neighbor (ANN) search**.

```
┌─────────────────────────────────────────────────────────────┐
│                    Vector Database                          │
│  ┌──────────────────────────────────────────────────────┐   │
│  │  ID  │  Embedding (1536 floats)  │  Text  │ Metadata │   │
│  ├──────┼───────────────────────────┼────────┼──────────┤   │
│  │  001 │  [0.12, -0.45, 0.87, ...] │ "doc1" │ {date}   │   │
│  │  002 │  [0.34, -0.22, 0.91, ...] │ "doc2" │ {date}   │   │
│  │  003 │  [-0.09, 0.67, 0.23, ...] │ "doc3" │ {date}   │   │
│  └──────┴───────────────────────────┴────────┴──────────┘   │
│                         ↑                                   │
│              ANN Index (HNSW / IVF)                         │
│              Finds nearest vectors in milliseconds          │
└─────────────────────────────────────────────────────────────┘

Query: "internet down" → embed → [0.11, -0.44, 0.88, ...]
                                        ↓
                          Find top-k nearest vectors
                                        ↓
                          Return: "doc1", "doc3" (most similar)
```

### ChromaDB — Our Vector Store

ChromaDB is an **open-source, embedded vector database** — it runs locally with no server needed, perfect for development and learning.

```python
import chromadb

client = chromadb.PersistentClient(path='./chroma_db')  # saves to disk
collection = client.create_collection('my_collection')

# Add documents
collection.add(documents=[...], embeddings=[...], ids=[...])

# Query
results = collection.query(query_embeddings=[...], n_results=3)
```

## 3. What is RAG (Retrieval-Augmented Generation)?

RAG combines a **retrieval system** (vector DB) with a **generation system** (LLM) to produce answers grounded in your own data.

```
Without RAG:                          With RAG:
                                      
User: "Who won 100m in 2020?"         User: "Who won 100m in 2020?"
           ↓                                      ↓
        LLM                            Embed query → search Vector DB
           ↓                                      ↓
   "I may not have that                Retrieved: '2020 Olympics results...'
    exact information"                            ↓
                                       Prompt: 'Context: {retrieved}\n{query}'
                                                  ↓
                                                LLM
                                                  ↓
                                       "Marcell Jacobs won the 100m"
```

### Why RAG?

| Problem with plain LLMs | RAG Solution |
|---|---|
| Knowledge cutoff (old data) | Inject current data at query time |
| Hallucination (making things up) | Grounded in retrieved facts |
| No access to private data | Works with your internal docs |
| Can't cite sources | Returns source document with answer |

In [ ]:
# ── Create the Olympics Dataset CSV ──
# We'll create a realistic Olympics dataset to practice RAG

import pandas as pd
import io

olympics_csv = """
Year,City,Country,Athlete,Event,Medal,Time_Score,Nationality
2020,Tokyo,Japan,Marcell Jacobs,100m Men,Gold,9.80,Italy
2020,Tokyo,Japan,Fred Kerley,100m Men,Silver,9.84,USA
2020,Tokyo,Japan,Andre De Grasse,100m Men,Bronze,9.89,Canada
2020,Tokyo,Japan,Elaine Thompson-Herah,100m Women,Gold,10.61,Jamaica
2020,Tokyo,Japan,Shelly-Ann Fraser-Pryce,100m Women,Silver,10.74,Jamaica
2020,Tokyo,Japan,Shericka Jackson,100m Women,Bronze,10.76,Jamaica
2020,Tokyo,Japan,Neeraj Chopra,Javelin Throw Men,Gold,87.58m,India
2020,Tokyo,Japan,Milkha Singh Tribute,400m Men,—,—,India
2020,Tokyo,Japan,Karsten Warholm,400m Hurdles Men,Gold,45.94 WR,Norway
2020,Tokyo,Japan,Sydney McLaughlin,400m Hurdles Women,Gold,51.46 WR,USA
2020,Tokyo,Japan,Caeleb Dressel,100m Freestyle Swimming,Gold,47.02,USA
2020,Tokyo,Japan,Adam Peaty,100m Breaststroke Men,Gold,57.37,UK
2020,Tokyo,Japan,Ariarne Titmus,400m Freestyle Women,Gold,3:56.69,Australia
2020,Tokyo,Japan,USA Basketball,Basketball Men,Gold,—,USA
2020,Tokyo,Japan,USA Softball,Softball Women,Silver,—,USA
2016,Rio,Brazil,Usain Bolt,100m Men,Gold,9.81,Jamaica
2016,Rio,Brazil,Usain Bolt,200m Men,Gold,19.78,Jamaica
2016,Rio,Brazil,Elaine Thompson,100m Women,Gold,10.71,Jamaica
2016,Rio,Brazil,Michael Phelps,200m Individual Medley,Gold,1:54.66,USA
2016,Rio,Brazil,Simone Biles,Artistic Gymnastics All-around,Gold,62.198,USA
2016,Rio,Brazil,PV Sindhu,Badminton Women,Silver,—,India
2016,Rio,Brazil,Sakshi Malik,Wrestling Women 58kg,Bronze,—,India
2012,London,UK,Usain Bolt,100m Men,Gold,9.63 OR,Jamaica
2012,London,UK,Mo Farah,5000m Men,Gold,13:41.66,UK
2012,London,UK,Mo Farah,10000m Men,Gold,27:30.42,UK
2012,London,UK,Jessica Ennis-Hill,Heptathlon Women,Gold,6955 pts,UK
2012,London,UK,Michael Phelps,100m Butterfly,Gold,51.21,USA
2012,London,UK,Saina Nehwal,Badminton Women,Bronze,—,India
2012,London,UK,Vijay Kumar,10m Air Pistol Shooting,Silver,—,India
2008,Beijing,China,Michael Phelps,200m Butterfly,Gold,1:52.03 WR,USA
2008,Beijing,China,Usain Bolt,100m Men,Gold,9.69 WR,Jamaica
2008,Beijing,China,Abhinav Bindra,10m Air Rifle Shooting,Gold,—,India
"""

df = pd.read_csv(io.StringIO(olympics_csv.strip()))
df.to_csv('olympics_data.csv', index=False)

print('✅ Olympics dataset created: olympics_data.csv')
print(f'   Rows: {len(df)}')
print(f'   Columns: {list(df.columns)}')
print(f'   Years covered: {sorted(df["Year"].unique())}')
print(f'\nSample records:')
print(df.head(6).to_string(index=False))

## 🔬 Lab 1: CSV → ChromaDB Vector Database

In this lab we will:
1. Load the Olympics CSV
2. Create meaningful text descriptions from each row
3. Generate embeddings
4. Store everything in ChromaDB with metadata

In [ ]:
# ── Lab 1: Ingest CSV into ChromaDB ──

import chromadb
import pandas as pd
import shutil, os

# Clean slate for demo
if os.path.exists('./chroma_olympics'):
    shutil.rmtree('./chroma_olympics')

# Load dataset
df = pd.read_csv('olympics_data.csv')

# Step 1: Create rich text description from each CSV row
def row_to_text(row):
    """Convert a CSV row into a natural language description for embedding."""
    medal = row['Medal'] if pd.notna(row['Medal']) and row['Medal'] != '—' else 'participated'
    score = f"with result {row['Time_Score']}" if row['Time_Score'] != '—' else ''
    return (
        f"{row['Athlete']} from {row['Nationality']} won the {medal} medal "
        f"in {row['Event']} at the {row['Year']} {row['City']} Olympics "
        f"{score}."
    )

df['text'] = df.apply(row_to_text, axis=1)
print('📝 Sample text descriptions:')
for t in df['text'].head(4):
    print(f'  • {t}')

# Step 2: Initialize ChromaDB with OpenAI embedding function
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

embedding_fn = OpenAIEmbeddingFunction(
    api_key=os.getenv('OPENAI_API_KEY'),
    model_name='text-embedding-3-small'
)

chroma_client = chromadb.PersistentClient(path='./chroma_olympics')
collection = chroma_client.create_collection(
    name='olympics',
    embedding_function=embedding_fn
)

# Step 3: Add all records to ChromaDB
print('\n⚙️  Adding documents to ChromaDB...')
batch_size = 10
for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size]
    collection.add(
        ids=[f'rec_{j}' for j in batch.index],
        documents=batch['text'].tolist(),
        metadatas=[
            {
                'year':        int(row['Year']),
                'athlete':     row['Athlete'],
                'event':       row['Event'],
                'medal':       str(row['Medal']),
                'nationality': row['Nationality'],
                'city':        row['City'],
            }
            for _, row in batch.iterrows()
        ]
    )
    print(f'  Added batch {i//batch_size + 1} ({min(i+batch_size, len(df))}/{len(df)} records)')

print(f'\n✅ Vector database built!')
print(f'   Collection: {collection.name}')
print(f'   Documents:  {collection.count()}')
print(f'   Saved to:   ./chroma_olympics/')

## 🔬 Lab 2: Query & Retrieve Answers from Vector DB

In [ ]:
# ── Lab 2: Querying the Vector Database ──

def search_db(query, n_results=3, filters=None):
    """Search the vector DB and display results."""
    kwargs = {'query_texts': [query], 'n_results': n_results}
    if filters:
        kwargs['where'] = filters
    results = collection.query(**kwargs)
    
    print(f'\n🔍 Query: "{query}"')
    if filters:
        print(f'   Filter: {filters}')
    print('-' * 60)
    
    for i, (doc, dist, meta) in enumerate(
        zip(results['documents'][0],
            results['distances'][0],
            results['metadatas'][0]), 1):
        similarity = 1 - dist  # convert distance to similarity
        print(f'  {i}. [{similarity:.3f}] {doc}')
        print(f'     {meta}')
    
    return results

# ── Different types of queries ──

print('=' * 60)
print('SEMANTIC SEARCH EXAMPLES')
print('=' * 60)

# Semantic synonyms
search_db("fastest sprinter on the track", n_results=3)

# Nationality filter
search_db("gold medalist swimmer", n_results=3)

# Indian athletes
search_db("Indian athletes who won medals", n_results=5,
          filters={'nationality': {'$eq': 'India'}})

# Year filter
search_db("records broken at the Olympics", n_results=3,
          filters={'year': {'$eq': 2020}})

In [ ]:
# ── Semantic Search vs Keyword Search Comparison ──

import re

def keyword_search(query, documents, n_results=3):
    """Simple keyword matching — traditional search."""
    words = set(query.lower().split())
    scores = []
    for doc in documents:
        doc_words = set(doc.lower().split())
        overlap = len(words & doc_words) / len(words)
        scores.append((overlap, doc))
    return sorted(scores, reverse=True)[:n_results]

all_docs = df['text'].tolist()

test_query = "athletes who run really fast"

print(f'Query: "{test_query}"')
print('\n❌ Keyword Search (traditional):')
print('-' * 50)
for score, doc in keyword_search(test_query, all_docs):
    print(f'  [{score:.2f}] {doc[:70]}')

print('\n✅ Semantic/Vector Search:')
print('-' * 50)
results = collection.query(query_texts=[test_query], n_results=3)
for doc, dist in zip(results['documents'][0], results['distances'][0]):
    print(f'  [{1-dist:.3f}] {doc[:70]}')

print('\n💡 Vector search finds relevant results even with different words!')

## 🔬 Lab 3: Full RAG Pipeline with LangChain

Now we build the complete RAG pipeline:
1. Receive a natural language question
2. Retrieve top-k relevant documents from ChromaDB
3. Construct a prompt with the retrieved context
4. Generate a grounded answer with GPT-4o-mini
5. Return the answer with source citations

In [ ]:
# ── Lab 3: RAG Pipeline (from scratch, no framework) ──

class RAGPipeline:
    """
    A complete Retrieval-Augmented Generation pipeline.
    Step 1: Retrieve relevant documents from ChromaDB
    Step 2: Augment the prompt with retrieved context
    Step 3: Generate grounded answer with LLM
    """
    
    def __init__(self, chroma_collection, openai_client, model='gpt-4o-mini', top_k=4):
        self.collection = chroma_collection
        self.client     = openai_client
        self.model      = model
        self.top_k      = top_k
    
    def retrieve(self, query, filters=None):
        """Step 1: Retrieve relevant documents."""
        kwargs = {'query_texts': [query], 'n_results': self.top_k}
        if filters:
            kwargs['where'] = filters
        return self.collection.query(**kwargs)
    
    def build_prompt(self, query, retrieved_results):
        """Step 2: Augment prompt with context."""
        context_parts = []
        for i, (doc, meta) in enumerate(
            zip(retrieved_results['documents'][0],
                retrieved_results['metadatas'][0]), 1):
            context_parts.append(f'Source {i}: {doc}')
        
        context = '\n'.join(context_parts)
        
        return f"""You are an Olympics knowledge assistant. Answer the question using ONLY 
the provided context. If the answer is not in the context, say "I don't have that information."
Always mention which source(s) you used.

Context:
{context}

Question: {query}

Answer:"""
    
    def generate(self, augmented_prompt):
        """Step 3: Generate answer with LLM."""
        response = self.client.chat.completions.create(
            model=self.model,
            messages=[{'role': 'user', 'content': augmented_prompt}],
            temperature=0.0,
            max_tokens=300
        )
        return response.choices[0].message.content
    
    def ask(self, question, filters=None, verbose=True):
        """Run the full RAG pipeline for a question."""
        # Retrieve
        results = self.retrieve(question, filters)
        
        # Augment
        augmented = self.build_prompt(question, results)
        
        # Generate
        answer = self.generate(augmented)
        
        if verbose:
            print(f'\n❓ Question: {question}')
            if filters:
                print(f'   Filter: {filters}')
            print(f'\n📚 Retrieved {len(results["documents"][0])} sources:')
            for i, (doc, dist) in enumerate(
                zip(results['documents'][0], results['distances'][0]), 1):
                print(f'   {i}. [{1-dist:.3f}] {doc[:70]}...' if len(doc)>70 else f'   {i}. [{1-dist:.3f}] {doc}')
            print(f'\n✅ Answer:')
            print(answer)
            print()
        
        return answer

rag = RAGPipeline(collection, client)
print('✅ RAG Pipeline initialized')

In [ ]:
# ── Run the RAG Pipeline ──

print('🏅 OLYMPICS RAG ASSISTANT')
print('=' * 60)

questions = [
    "Who won the 100m sprint at the 2020 Tokyo Olympics?",
    "Which Indian athletes won medals at the Olympics and in what events?",
    "Which world records were broken at the Tokyo Olympics?",
    "Tell me about Michael Phelps' achievements.",
    "Who dominated the sprinting events across multiple Olympics?",
]

for q in questions:
    rag.ask(q)
    print('-' * 60)

In [ ]:
# ── LangChain RAG: Using the Framework ──
# LangChain provides pre-built components for RAG pipelines

from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.text_splitter import RecursiveCharacterTextSplitter
import shutil

# Clean existing store
if os.path.exists('./chroma_lc'):
    shutil.rmtree('./chroma_lc')

# LangChain embedding + vectorstore
embeddings = OpenAIEmbeddings(model='text-embedding-3-small')

# Add documents from DataFrame
texts    = df['text'].tolist()
metadata = df[['Year','Athlete','Event','Medal','Nationality','City']].to_dict('records')

vectorstore = Chroma.from_texts(
    texts=texts,
    embedding=embeddings,
    metadatas=metadata,
    persist_directory='./chroma_lc'
)

retriever = vectorstore.as_retriever(search_kwargs={'k': 4})

# LangChain RAG prompt template
RAG_TEMPLATE = """
You are an expert Olympics analyst. Answer the question based strictly on the context below.
If the information is not in the context, respond with "Information not available."
Be specific and mention athlete names, years, and results.

Context:
{context}

Question: {question}

Answer:
"""

prompt_template = ChatPromptTemplate.from_template(RAG_TEMPLATE)
llm = ChatOpenAI(model='gpt-4o-mini', temperature=0)

def format_docs(docs):
    return '\n'.join([d.page_content for d in docs])

# Build LangChain RAG chain using LCEL (LangChain Expression Language)
rag_chain = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt_template
    | llm
    | StrOutputParser()
)

print('✅ LangChain RAG chain built')
print('\n🔗 Chain: retriever → prompt → LLM → parser')
print()

# Test
test_q = "Which country dominated sprinting at the 2016 and 2020 Olympics?"
print(f'❓ {test_q}')
print('✅', rag_chain.invoke(test_q))

In [ ]:
# ── Interactive Q&A Session ──

qa_pairs = [
    "Who won gold in javelin throw at the 2020 Olympics?",
    "List all gold medalists from Jamaica.",
    "Compare the 100m men results between 2012, 2016, and 2020.",
    "Which events had world records broken?",
    "Who is Simone Biles and what did she win?",
]

print('🏅 LangChain RAG: Q&A Session')
print('=' * 60)

for q in qa_pairs:
    answer = rag_chain.invoke(q)
    print(f'\n❓ {q}')
    print(f'✅ {answer}')

In [ ]:
# ── ✏️  Your Turn: Ask Your Own Questions ──
# Use the RAG system to answer any Olympics question

your_question = "Which athlete do you find most impressive and why?"

print(f'❓ Your question: {your_question}')
print('✅ Answer:', rag_chain.invoke(your_question))

# Also try the custom RAG pipeline:
print('\n--- Custom RAG Pipeline ---')
rag.ask(your_question)

In [ ]:
# ── RAG Architecture Summary Visualization ──

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.patches import FancyArrowPatch

fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 7)
ax.axis('off')
ax.set_title('RAG Pipeline Architecture', fontsize=15, fontweight='bold')

def box(ax, x, y, w, h, fc, ec, text, fontsize=9, text_color='white', bold=True):
    ax.add_patch(mpatches.FancyBboxPatch((x, y), w, h, boxstyle='round,pad=0.15', fc=fc, ec=ec, lw=1.5))
    ax.text(x+w/2, y+h/2, text, ha='center', va='center', fontsize=fontsize,
            color=text_color, fontweight='bold' if bold else 'normal', wrap=True)

def arrow(ax, x1, y1, x2, y2, color='#555'):
    ax.annotate('', xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle='->', color=color, lw=2))

# Offline (indexing) pipeline
box(ax, 0.3, 5.0, 2.2, 1.2, '#D6E4F0', '#1F4E79', 'Raw Documents\n(CSV, PDFs, web)', text_color='#1F4E79')
box(ax, 3.0, 5.0, 2.2, 1.2, '#1F4E79', '#1F4E79', 'Chunk & Embed\n(split + vectorize)')
box(ax, 5.8, 5.0, 2.2, 1.2, '#2E75B6', '#2E75B6', 'Vector DB\n(ChromaDB)')

arrow(ax, 2.5, 5.6,  3.0, 5.6)
arrow(ax, 5.2, 5.6,  5.8, 5.6)

ax.text(4.0, 6.5, '── Offline Indexing ──', ha='center', fontsize=10, color='#1F4E79', style='italic')

# Online (query) pipeline
box(ax, 0.3, 2.5, 2.2, 1.2, '#D6EAD6', '#1B5E3B', 'User Question', text_color='#1B5E3B')
box(ax, 3.0, 2.5, 2.2, 1.2, '#1B5E3B', '#1B5E3B', 'Embed Query')
box(ax, 5.8, 2.5, 2.2, 1.2, '#239B56', '#239B56', 'Similarity\nSearch (top-k)')
box(ax, 8.6, 2.5, 2.2, 1.2, '#1B5E3B', '#1B5E3B', 'Augment Prompt\nwith Context')
box(ax, 11.3, 2.5, 2.2, 1.2, '#D6EAD6', '#1B5E3B', 'Grounded\nAnswer', text_color='#1B5E3B')

arrow(ax, 2.5, 3.1, 3.0, 3.1)
arrow(ax, 5.2, 3.1, 5.8, 3.1)
arrow(ax, 8.0, 3.1, 8.6, 3.1)
arrow(ax, 10.7, 3.1, 11.3, 3.1, '#888')

ax.text(7.0, 1.8, '── Online Query ──', ha='center', fontsize=10, color='#1B5E3B', style='italic')

# Vertical arrow from vector DB to search
ax.annotate('', xy=(6.9, 3.7), xytext=(6.9, 5.0),
            arrowprops=dict(arrowstyle='->', color='#888', lw=1.5, linestyle='dashed'))
ax.text(7.5, 4.3, 'retrieve', fontsize=9, color='#888', style='italic')

# LLM box
box(ax, 9.7, 0.5, 2.5, 1.2, '#7B3F00', '#7B3F00', 'LLM\n(GPT-4o-mini)')
ax.annotate('', xy=(9.7+1.25, 2.5), xytext=(9.7+1.25, 1.7),
            arrowprops=dict(arrowstyle='->', color='#7B3F00', lw=1.5))

plt.tight_layout()
plt.savefig('rag_architecture.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ RAG architecture diagram saved')

## ✅ Session 4 Summary

| Concept | Key Takeaway |
|---|---|
| **Embeddings** | Dense vectors that encode semantic meaning — similar text = close vectors |
| **Vector DB** | Stores embeddings + content + metadata, enables fast ANN search |
| **ChromaDB** | Local, open-source vector DB — no server needed |
| **Cosine similarity** | Measures angle between vectors — 1.0 = identical |
| **RAG** | Retrieve → Augment → Generate — grounds LLM in your data |
| **LangChain** | Framework that chains retriever + prompt + LLM + parser |
| **When to use RAG** | Private data, current info, reducing hallucination |

## 🔗 What's Next
**Day 2 — Session 5** introduces **AI Agents** — systems that don't just answer questions but take actions, use tools, and reason across multiple steps. We'll use everything built in Day 1 as building blocks.

## 🧹 Cleanup (optional)
```python
import shutil
shutil.rmtree('./chroma_olympics')
shutil.rmtree('./chroma_lc')
```

---
*Agentic AI Nano Bootcamp | Day 1 Session 4*